In [ ]:
# S3 prefix
prefix = "DEMO-scikit-byo-iris"

# Define IAM role
import boto3
import re

import os
import numpy as np
import pandas as pd
from sagemaker import get_execution_role

role = get_execution_role()

In [ ]:
import sagemaker as sage
from time import gmtime, strftime

sess = sage.Session()

In [ ]:
from time import strftime, gmtime

WORK_DIRECTORY = "/home/sagemaker-user/Emi-y-Mora/data/prep"

default_bucket = sess.default_bucket()
default_bucket_prefix = sess.default_bucket_prefix

prefix = f"emi-y-mora-prep-{strftime('%Y%m%d-%H%M%S', gmtime())}"

if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

data_location = sess.upload_data(
    path=WORK_DIRECTORY,
    bucket=default_bucket,
    key_prefix=prefix,
)

print(data_location)

In [ ]:
import boto3

s3 = boto3.client("s3")
resp = s3.list_objects_v2(Bucket=default_bucket, Prefix=prefix)

for obj in resp.get("Contents", []):
    print(obj["Key"])

In [ ]:
account = sess.boto_session.client("sts").get_caller_identity()["Account"]
region = sess.boto_session.region_name

image = "635026339135.dkr.ecr.us-east-1.amazonaws.com/emi-y-mora-train:latest"

In [ ]:
default_bucket = sess.default_bucket()

s3_output_path = f"s3://{default_bucket}/emi-y-mora-output"

In [ ]:
#esta es la buena cawn
tree = sage.estimator.Estimator(
    image,
    role,
    1,
    "ml.m5.large",
    output_path=s3_output_path,
    sagemaker_session=sess,
    hyperparameters={
        "prep-dir": "/opt/ml/input/data/train"
    },
)

tree.fit({"train": data_location})

In [ ]:
print(tree.latest_training_job.describe()["TrainingJobStatus"])

In [ ]:
model_data = tree.model_data
print(model_data)

In [ ]:
from sagemaker.model import Model

inference_image = "635026339135.dkr.ecr.us-east-1.amazonaws.com/emi-y-mora-inference:latest"

serving_model = Model(
    image_uri=inference_image,
    model_data=model_data,
    role=role,
    sagemaker_session=sess,
)


In [ ]:
predictor = serving_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="emi-y-mora-endpoint"
)

In [ ]:
import boto3

sm = boto3.client("sagemaker")

sm.describe_endpoint(
    EndpointName="emi-y-mora-endpoint"
)["EndpointStatus"]

In [ ]:
from sagemaker.predictor import Predictor

predictor = Predictor(
    endpoint_name="emi-y-mora-endpoint",
    sagemaker_session=sess
)

In [ ]:
import pandas as pd

sample = pd.read_csv("/home/sagemaker-user/Emi-y-Mora/data/prep/X_valid.csv").head(5)
payload = sample.to_csv(index=False, header=True)

result = predictor.predict(
    payload,
    initial_args={"ContentType": "text/csv"}
)

print(result)

In [ ]:
#PARA BORRAR EL ENDPOINT 
predictor.delete_endpoint()